# Data Refresh Workflow

> Goal: Compare a newly downloaded Oracle's Elixir 2026 file against the previous raw snapshot, quantify dataset growth, and record a promotion decision before rerunning the pipeline.

## Steps

1. Load old and newly downloaded raw datasets.
2. Compare total row counts.
3. Compare `datacompleteness == "complete"` row counts.
4. Compare team-level row counts.
5. Compare unique team-level match counts (`gameid`).
6. Compare covered date ranges.
7. Record observations and promotion decision.

In [7]:
import pandas as pd

old_path = '../data/raw/2026_LoL_OraclesElixir_original.csv'
new_path = '../data/raw/2026_LoL_OraclesElixir.csv'

old_df = pd.read_csv(old_path, low_memory=False)
new_df = pd.read_csv(new_path, low_memory=False)

print('Old rows:', len(old_df))
print('New rows:', len(new_df))
print('Difference:', len(new_df) - len(old_df))

Old rows: 34260
New rows: 55992
Difference: 21732


## 1) Load Raw Files and Compare Total Rows

### Result
- **Old rows:** 34,260
- **New rows:** 49,788
- **Difference:** +15,528

In [8]:
old_complete = old_df[old_df['datacompleteness'] == 'complete']
new_complete = new_df[new_df['datacompleteness'] == 'complete']

print('Old complete rows:', len(old_complete))
print('New complete rows:', len(new_complete))
print('Difference:', len(new_complete) - len(old_complete))

Old complete rows: 31332
New complete rows: 51432
Difference: 20100


## 2) Compare Complete Rows

### Result
- **Old complete rows:** 31,332
- **New complete rows:** 45,780
- **Difference:** +14,448

In [9]:
old_team = old_complete[old_complete['position'] == 'team'].copy()
new_team = new_complete[new_complete['position'] == 'team'].copy()

print('Old team rows:', len(old_team))
print('New team rows:', len(new_team))
print('Difference:', len(new_team) - len(old_team))

Old team rows: 5222
New team rows: 8572
Difference: 3350


## 3) Compare Team-Level Rows

### Result
- **Old team rows:** 5,222
- **New team rows:** 7,630
- **Difference:** +2,408

In [10]:
old_matches = old_team['gameid'].nunique()
new_matches = new_team['gameid'].nunique()

print('Old matches:', old_matches)
print('New matches:', new_matches)
print('Difference:', new_matches - old_matches)

Old matches: 2611
New matches: 4286
Difference: 1675


## 4) Compare Unique Match Counts

### Result
- **Old unique matches (`gameid`):** 2,611
- **New unique matches (`gameid`):** 3,815
- **Difference:** +1,204

In [11]:
old_team['date'] = pd.to_datetime(old_team['date'])
new_team['date'] = pd.to_datetime(new_team['date'])

print('Old date range:', old_team['date'].min(), 'to', old_team['date'].max())
print('New date range:', new_team['date'].min(), 'to', new_team['date'].max())

Old date range: 2026-01-08 17:08:27 to 2026-04-11 17:11:53
New date range: 2026-01-08 17:08:27 to 2026-05-16 18:14:47


## 5) Compare Date Coverage

### Result
- **Old date range (team-level complete rows):** 2026-01-08 17:08:27 to 2026-04-11 17:11:53
- **New date range (team-level complete rows):** 2026-01-08 17:08:27 to 2026-05-16 18:14:47
- **Coverage extension:** +35 days on the upper bound

### Interpretation
The refreshed file extends the upper date bound to mid-May while preserving the lower bound, indicating additive growth.

## 6) Data Refresh Log

_Updated: 2026-05-16_

### Current Comparison Summary

| Metric | Old | New | Δ |
|---|---:|---:|---:|
| Raw rows | 34,260 | 49,788 | +15,528 |
| Complete rows | 31,332 | 45,780 | +14,448 |
| Team-level rows | 5,222 | 7,630 | +2,408 |
| Unique matches (`gameid`) | 2,611 | 3,815 | +1,204 |

### Date Range (team-level complete rows)
- **Old:** 2026-01-08 17:08:27 to 2026-04-11 17:11:53
- **New:** 2026-01-08 17:08:27 to 2026-05-16 18:14:47

### Decision
Keep `2026_LoL_OraclesElixir.csv` as the primary raw dataset for the next pipeline run.

### Notes
- The current primary file is additive compared with the original snapshot.
- Keep `2026_LoL_OraclesElixir_original.csv` for reproducibility and rollback.